# Experimento 1: Sensibilidad al Tamaño de la Ventana (Retardo vs Resolución)

El objetivo de este cuadernillo es evaluar el efecto de variar el tamaño de la ventana (W) en la exactitud del sistema, comparando los resultados para el músculo Flexor (FDS) y Extensor (ED).

In [ ]:
import sys
import os
# Añadir el directorio padre al path para importar 'src'
sys.path.append(os.path.abspath('..'))

import matplotlib.pyplot as plt
from src import dataset, features
from src.models import rf

In [ ]:
window_sizes = [100, 150, 200, 250, 300, 400]
muscles = ['FDS', 'ED']
subject = 'S01'

results = {'FDS': [], 'ED': []}

for muscle in muscles:
    print(f"\n=== Evaluando Músculo: {muscle} ===")
    for w in window_sizes:
        print(f"Procesando ventana de {w} ms...")
        
        # 1. Cargar tensores crudos
        X_train_raw, y_train, X_test_raw, y_test = dataset.load_and_segment_dataset(
            subject=subject,
            muscle=muscle,
            window_size=w,
            overlap=w//2
        )
        
        # 2. Extraer características
        X_train_feat = features.extract_features(X_train_raw)
        X_test_feat = features.extract_features(X_test_raw)
        
        # 3. Escalar características
        X_train_scaled, X_test_scaled, scaler = features.scale_features(X_train_feat, X_test_feat)
        
        # 4. Entrenar y evaluar modelo (usaremos Random Forest como base rápida)
        model = rf.train(X_train_scaled, y_train)
        acc, report = rf.evaluate(model, X_test_scaled, y_test)
        
        print(f"  -> Exactitud: {acc * 100:.2f}%")
        results[muscle].append(acc)


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(window_sizes, results['FDS'], marker='o', label='FDS (Flexor)')
plt.plot(window_sizes, results['ED'], marker='s', label='ED (Extensor)')

plt.title('Trade-off: Retardo (Tamaño de Ventana) vs Exactitud')
plt.xlabel('Tamaño de Ventana (ms)')
plt.ylabel('Exactitud (Accuracy)')
plt.legend()
plt.grid(True)
plt.show()
